In [1]:
import pandas as pd
import numpy as np
import requests
import os
from bs4 import BeautifulSoup
from tqdm import tqdm

In [2]:
taxo_planktoscope = pd.read_csv("/lustre/fswork/projects/rech/tec/uod68bo/am/planktonzilla/notebooks/meta_planktoscope.csv")

In [3]:
import re

def get_last_capitalized(lineage: str):
    if not isinstance(lineage, str):
        return None

    # separar jerarquía por ">"
    parts = lineage.split(">")

    # quedarse solo con los niveles que empiezan con mayúscula
    caps = [p for p in parts if re.match(r"^[A-Z]", p)]

    return caps[-1] if caps else None


In [4]:
taxo_planktoscope['last_taxon'] = taxo_planktoscope['annotation_hierarchy'].apply(get_last_capitalized)

In [5]:
taxon_dict = (
    taxo_planktoscope[['annotation_category', 'last_taxon']]
    .drop_duplicates(subset=['annotation_category'])
    .set_index('annotation_category')['last_taxon']
    .to_dict()
)

In [ ]:
import re
import numpy as np

RANK_MAP = {
    "Kingdom": "kingdom",
    "Phylum": "phylum",
    "Class": "class",
    "Order": "order",
    "Family": "family",
    "Genus": "genus",
    "Species": "species",
}

def fill_taxonomy(df, idx, taxonomy_str):
    # Inicializar columnas con NaN
    for col in RANK_MAP.values():
        df.loc[idx, col] = np.nan
    
    # Regex para extraer "Nombre (Rank)"
    pattern = r"([^()]+)\s*\(([^()]+)\)"
    
    for name, rank in re.findall(pattern, taxonomy_str):
        name = name.strip()
        rank = rank.strip()
        
        if rank in RANK_MAP:
            df.loc[idx, RANK_MAP[rank]] = name
    
    # flags extra
    df.loc[idx, "taxonomy"] = "yes"
    df.loc[idx, "status"] = "accepted"

## Rellenar cosas de Ecotaxa

- Zooscan -> 120 clases (listo)
- ZooCAMNet -> 93 clases (listo)
- UVP6Net -> 54 clases (listo)
- Flowcamnet -> 38 clases (listo)
- ISIISNet -> 32 clases (listo)


In [ ]:
# Esto es para sacar un ejemplo de cada folder de clase
# y crea un diccionario del tipo {label: objid}

# path_zoo = "/lustre/fsn1/projects/rech/tec/uod68bo/data/isiisnetdatasetimporter_imagefolder"

# result = {}

# for subfolder in os.listdir(path_zoo):
#     subfolder_path = os.path.join(path_zoo, subfolder)
    
#     if os.path.isdir(subfolder_path):
#         # listar archivos de la subcarpeta
#         files = os.listdir(subfolder_path)
        
#         # filtrar solo imágenes (opcional pero recomendable)
#         images = [f for f in files if f.lower().endswith(('.png', '.jpg', '.jpeg', '.tif', '.tiff'))]
        
#         if images:
#             # tomar la primera imagen
#             img_name = os.path.splitext(images[0])[0]
#             result[subfolder] = img_name


In [ ]:
def get_ecotaxa_taxonomy(obj_id, timeout=10):

    '''
    recibe el objid de ecotaxa y retorna un diccionario:
    {
    first: clase asociada (puede que sea incorrecta/ que no exista en WoRMS)
    hierarchy: taxonomia pseudo-asociada de WoRMS (puede que le añadan cosas al final como larvae, egg, etc)
    chosen_taxon: ultimo nivel taxonomico 100% confiable que podemos usar para ver en WoRMS
    }
    '''


    url = f"https://ecotaxa.obs-vlfr.fr/objectdetails/{obj_id}"
    
    r = requests.get(url, timeout=timeout)
    r.raise_for_status()

    soup = BeautifulSoup(r.text, "html.parser")
    text = soup.get_text("\n")

    # limpiar líneas
    lines = [l.strip().replace("\u2003", "") for l in text.split("\n")]
    lines = [l for l in lines if l]

    first = None
    hierarchy = None

    # buscar bloque Classification
    for i, line in enumerate(lines):
        if line.startswith("Classification"):
            if i + 1 < len(lines):
                first = lines[i + 1]
            if i + 2 < len(lines):
                hierarchy = lines[i + 2]
            break

    # normalizar first si viene con "<"
    if first and "<" in first:
        first = first.split("<")[0].strip()

    # regla: si first está en minúsculas -> tomar segundo taxón
    chosen = first
    if first and hierarchy and first[0].islower():
        parts = [p.strip() for p in hierarchy.split("<")]

        # seleccionar la primera palabra que empiece con mayúscula
        for p in parts:
            if p and p[0].isupper():
                chosen = p
                break

    return {
        "first": first,
        "hierarchy": hierarchy,
        "chosen_taxon": chosen,
    }


In [ ]:
def complete_row(label, dataset, raw_label):

    '''
    recibe un nivel taxonomico y retorna una fila con la taxonomia asociada consultada a WoRMS
    '''

    data = {}  

    url = f"https://www.marinespecies.org/rest/AjaxAphiaRecordsByNamePart/{label}"
    
    try:
        response = requests.get(url)
        records = response.json()

        if records:
            for rec in records:
                aphia_id = rec.get("id", None)
                if aphia_id is None:
                    continue

                url_id = f"https://www.marinespecies.org/rest/AphiaRecordByAphiaID/{aphia_id}"
                response_id = requests.get(url_id)
                record_full = response_id.json()

                status = record_full.get("status", "").lower()

                if status == "accepted":
                    data = record_full
                    break
                
    except Exception:
        data = {}

    # Función auxiliar para extraer taxonomía o np.nan
    def get_taxon(key):
        return data.get(key, np.nan) if isinstance(data, dict) else np.nan

    kingdom = get_taxon("kingdom")
    phylum  = get_taxon("phylum")
    class_  = get_taxon("class")
    order   = get_taxon("order")
    family  = get_taxon("family")
    genus   = get_taxon("genus")
    species = get_taxon("scientificname")  # WoRMS usa scientificname

    # Construir taxonomy solo si existe al menos un nivel taxonómico
    taxonomy_levels = [kingdom, phylum, class_, order, family, genus, species]
    if all(pd.isna(x) for x in taxonomy_levels):
        taxonomy = np.nan
    else:
        taxonomy = "yes"

    new_row = {
        "Dataset": dataset,
        "Raw_Labels": raw_label,
        "kingdom": kingdom,
        "phylum": phylum,
        "class": class_,
        "order": order,
        "family": family,
        "genus": genus,
        "species": species,
        "taxonomy": taxonomy,
        "status": get_taxon("status")
    }

    return new_row

In [ ]:
rows = []

for old_name, new_name in tqdm(taxon_dict.items(), total=len(taxon_dict)):
    new_row = complete_row(new_name, "planktoscope", old_name)
    rows.append(new_row)

df_taxonomy_new = pd.DataFrame(rows)

In [ ]:
x = pd.read_csv("/lustre/fswork/projects/rech/tec/uod68bo/am/planktonzilla/notebooks/planktonzilla_taxonomy_final.csv")

In [ ]:
cols_to_nan = df_taxonomy_new.columns.difference(["Raw_Labels", "Dataset"])

In [ ]:
idx = 228
taxonomy_str = "Chromista (Kingdom) Heterokontophyta (Phylum) Bacillariophytina (Subphylum) Bacillariophyceae (Class) Coscinodiscophycidae (Subclass) Rhizosolenianae (Superorder) Rhizosoleniales (Order) Rhizosoleniaceae (Family) Rhizosolenia (Genus)"
fill_taxonomy(df_taxonomy_new, idx, taxonomy_str)

In [ ]:
df = pd.read_csv("/lustre/fswork/projects/rech/tec/uod68bo/am/planktonzilla/notebooks/planktonzilla_taxonomy_final.csv")

In [ ]:
df[df["Species"]=="richelia"]

In [ ]:
taxo_planktoscope[(taxo_planktoscope["annotation_category"] == "Rhizosolenia inter. Richelia")].iloc[0]["annotation_hierarchy"]

In [ ]:
cols_to_nan = df_taxonomy_new.columns.difference(["Raw_Labels", "Dataset"])
df_taxonomy_new.loc[258, cols_to_nan] = np.nan

In [ ]:
#210

In [ ]:
df_taxonomy_new[240:]

In [ ]:
# rows = []

# for name in tqdm(result.keys(), total=len(result)):
#     objid = result[name]
#     pseudo_taxo = get_ecotaxa_taxonomy(objid)
#     new_row = complete_row(pseudo_taxo["chosen_taxon"], "isiisnet", name)

#     rows.append(new_row)

# df_taxonomy_new = pd.DataFrame(rows)

In [ ]:
#pennates = Chromista (Kingdom) Bacillariophyta (Phylum)

In [ ]:
idx = 73
taxonomy_str = "Chromista (Kingdom) Harosa (Subkingdom) Rhizaria (Infrakingdom) Radiozoa (Phylum)"

fill_taxonomy(df_taxonomy_new, idx, taxonomy_str)

In [ ]:
taxon_dict['artefact']

## Rellenar cosas que no son de Ecotaxa

- Medplanktonset -> 139 clases 
- WHOI -> 103 clases ✅
- Jedi -> 95 clases
- Zoolake -> 35 clases (en el paper tambien detallan las clases, corroborar con esa info)
- SYKE Zooscan 2024 -> 20 clases
- Lensless -> 10 clases

In [ ]:
import requests
import numpy as np

rows = []

for _, row in df_whoi.iterrows():

    label = row["Raw_Labels"]
    url = f"https://www.marinespecies.org/rest/AjaxAphiaRecordsByNamePart/{label}"
    
    try:
        response = requests.get(url)
        records = response.json()

        # Caso 1: no hay resultados
        if not records:
            data = {}
        else:
            aphia_id = records[0].get("id", None)

            if aphia_id is None:
                data = {}
            else:
                url = f"https://www.marinespecies.org/rest/AphiaRecordByAphiaID/{aphia_id}"
                response = requests.get(url)
                data = response.json()

    except Exception:
        data = {}

    # Función auxiliar para extraer taxonomía o np.nan
    def get_taxon(key):
        return data.get(key, np.nan) if isinstance(data, dict) else np.nan

    kingdom = get_taxon("kingdom")
    phylum  = get_taxon("phylum")
    class_  = get_taxon("class")
    order   = get_taxon("order")
    family  = get_taxon("family")
    genus   = get_taxon("genus")
    species = get_taxon("scientificname")  # WoRMS usa scientificname

    # Construir taxonomy solo si existe al menos un nivel taxonómico
    taxonomy_levels = [kingdom, phylum, class_, order, family, genus, species]
    if all(pd.isna(x) for x in taxonomy_levels):
        taxonomy = np.nan
    else:
        taxonomy = "yes"

    new_row = {
        "Dataset": row["Dataset"],
        "Raw_Labels": row["Raw_Labels"],
        "kingdom": kingdom,
        "phylum": phylum,
        "class": class_,
        "order": order,
        "family": family,
        "genus": genus,
        "species": species,
        "taxonomy": taxonomy,
        "status": get_taxon("status")
    }

    rows.append(new_row)

df_taxonomy = pd.DataFrame(rows)


In [ ]:
import re
import numpy as np

RANK_MAP = {
    "Kingdom": "kingdom",
    "Phylum": "phylum",
    "Class": "class",
    "Order": "order",
    "Family": "family",
    "Genus": "genus",
    "Species": "species",
}

def fill_taxonomy(df, idx, taxonomy_str):
    # Inicializar columnas con NaN
    for col in RANK_MAP.values():
        df.loc[idx, col] = np.nan
    
    # Regex para extraer "Nombre (Rank)"
    pattern = r"([^()]+)\s*\(([^()]+)\)"
    
    for name, rank in re.findall(pattern, taxonomy_str):
        name = name.strip()
        rank = rank.strip()
        
        if rank in RANK_MAP:
            df.loc[idx, RANK_MAP[rank]] = name
    
    # flags extra
    df.loc[idx, "taxonomy"] = "yes"
    df.loc[idx, "status"] = "accepted"


In [ ]:
idx = 32
taxonomy_str = "Chromista (Kingdom) Heterokontophyta (Phylum) Bacillariophytina (Subphylum) Bacillariophyceae (Class) Coscinodiscophycidae (Subclass) Rhizosolenianae (Superorder) Rhizosoleniales (Order) Rhizosoleniaceae (Family) Dactyliosolen (Genus)"

fill_taxonomy(df_taxonomy, idx, taxonomy_str)

## Process final

In [ ]:
def final_process(df):
    df_final = df.copy()

    # Reemplazamos None por np.nans
    df_final = df_final.replace({None: np.nan})

    # Las especies deben ser genus-species
    # por lo que eliminamos las que tengan solo 1 palabra
    df_final["species"] = (
        df_final["species"]
        .astype(str)
        .str.strip()
        .str.split()
        .apply(lambda x: x[1] if len(x) >= 2 else np.nan)
    )

    # Ponemos en minusculas las jerarquias
    cols = ["kingdom", "phylum", "class", "order", "family", "genus", "species"]
    df_final[cols] = df_final[cols].apply(
        lambda col: col.where(col.isna(), col.astype(str).str.lower())
    )

    # Agregamos la ultima jerarquia asociada
    df_final["proposed_label"] = df_final[cols].apply(lambda row: row.dropna().iloc[-1] if row.notna().any() else np.nan, axis=1)

    return df_final


In [ ]:
df_final = final_process(df_taxonomy_new)

In [ ]:
df_final

In [16]:
df_final.to_csv("planktoscope_corrected.csv",index=False)